# routes/selection

> Route handlers for selection queue management (remove, reorder, clear)

In [ ]:
#| default_exp routes.selection

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable
from pathlib import Path

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_fasthtml_file_browser.routes.handlers import FileBrowserRouters
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls
from cjm_transcription_source_select.utils import is_media_file, detect_file_type
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.components.stats_panel import render_stats_panel
from cjm_transcription_source_select.components.file_browser_panel import sync_browser_selection

DEBUG_REORDER = False

## OOB Checkbox Helper

When selection changes via remove/clear/toggle, send targeted OOB checkbox cell updates
instead of re-rendering the entire file browser panel.

In [ ]:
#| export
def _render_oob_checkboxes(
    fb_routers: FileBrowserRouters,  # File browser routers
    selected_files: list,            # Current selected files
    selected_folders: list,          # Current selected folders
    changed_paths: list,             # Paths whose checkbox state changed
) -> tuple:  # OOB cell elements for visible checkboxes
    """Sync selection state and return targeted checkbox OOBs for changed paths."""
    sync_browser_selection(fb_routers._fb_state_getter(), selected_files, selected_folders)
    return fb_routers.render_selection_oobs(changed_paths)

## Remove Handler

In [ ]:
#| export
def _handle_remove(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    fb_routers: FileBrowserRouters,  # File browser routers
    sess,  # FastHTML session
    key: str,  # Item key (file path) to remove
):  # OOB tuple (selection panel, checkbox OOBs, stats panel)
    """Remove a file from the selection by key."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])
    selected_folders = step_state.get("selected_folders", [])
    extraction_results = step_state.get("extraction_results", {})

    selected_files = [f for f in selected_files if f["path"] != key]
    extraction_results.pop(key, None)

    # Track changed paths for targeted checkbox OOBs
    changed_paths = [key]

    # Auto-deselect parent folder if no files from it remain
    parent = str(Path(key).parent)
    if parent in selected_folders:
        remaining = {f["path"] for f in selected_files}
        has_sibling = any(str(Path(p).parent) == parent for p in remaining)
        if not has_sibling:
            selected_folders = [f for f in selected_folders if f != parent]
            changed_paths.append(parent)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=selected_files,
        selected_folders=selected_folders,
        extraction_results=extraction_results,
        verified=False,
        committed_audio_paths=[],
    )

    selection_oob = render_selection_panel(selected_files, urls, extraction_results)
    selection_oob.attrs["hx-swap-oob"] = "outerHTML"
    checkbox_oobs = _render_oob_checkboxes(fb_routers, selected_files, selected_folders, changed_paths)
    stats_oob = render_stats_panel(selected_files, urls, extraction_results, verified=False)
    stats_oob.attrs["hx-swap-oob"] = "outerHTML"
    return (selection_oob, *checkbox_oobs, stats_oob)

## Reorder Handler

## Clear Handler

In [ ]:
#| export
async def _handle_reorder(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    request,  # FastHTML request
    sess,  # FastHTML session
):  # Rendered selection panel
    """Reorder selected files based on SortableJS drag or keyboard Shift+Arrow."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])

    form_data = await request.form()

    # Check if this is a keyboard reorder (has direction from vals_map)
    direction = form_data.get("direction", "")
    focused_key = form_data.get("key", "")

    if direction and focused_key:
        # Keyboard reorder: swap item with neighbor
        result = list(selected_files)
        idx = next((i for i, f in enumerate(result) if f["path"] == focused_key), None)
        if idx is not None:
            if direction == "up" and idx > 0:
                result[idx], result[idx - 1] = result[idx - 1], result[idx]
            elif direction == "down" and idx < len(result) - 1:
                result[idx], result[idx + 1] = result[idx + 1], result[idx]
        reordered = result
    else:
        # Drag-drop reorder: rebuild list in new DOM order
        new_order_paths = form_data.getlist("item")

        if DEBUG_REORDER:
            print(f"\n[REORDER DEBUG] Handler called")
            print(f"  Content-Type: {request.headers.get('content-type', 'MISSING')}")
            print(f"  form_data keys: {list(form_data.keys())}")
            print(f"  form_data.getlist('item'): {new_order_paths}")
            print(f"  len(new_order_paths): {len(new_order_paths)}")
            print(f"  len(selected_files): {len(selected_files)}")

        path_to_file = {f["path"]: f for f in selected_files}
        reordered = [path_to_file[p] for p in new_order_paths if p in path_to_file]

        # Append any files not in the form data (safety fallback)
        reordered_paths = {f["path"] for f in reordered}
        for f in selected_files:
            if f["path"] not in reordered_paths:
                reordered.append(f)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=reordered,
    )

    return render_selection_panel(reordered, urls)

In [ ]:
#| export
def _handle_clear(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    fb_routers: FileBrowserRouters,  # File browser routers
    sess,  # FastHTML session
):  # OOB tuple (selection panel, checkbox OOBs, stats panel)
    """Clear all selected files and folders."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)

    # Capture paths before clearing for targeted checkbox OOBs
    old_files = step_state.get("selected_files", [])
    old_folders = step_state.get("selected_folders", [])
    changed_paths = [f["path"] for f in old_files] + list(old_folders)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=[],
        selected_folders=[],
        extraction_results={},
        verified=False,
        committed_audio_paths=[],
    )

    selection_oob = render_selection_panel([], urls)
    selection_oob.attrs["hx-swap-oob"] = "outerHTML"
    checkbox_oobs = _render_oob_checkboxes(fb_routers, [], [], changed_paths)
    stats_oob = render_stats_panel([], urls)
    stats_oob.attrs["hx-swap-oob"] = "outerHTML"
    return (selection_oob, *checkbox_oobs, stats_oob)

## Toggle All Handler

Bulk toggle all media files in the file browser's current directory. If any are unselected, adds them all. If all are already selected, removes them all.

In [ ]:
#| export
from cjm_transcription_source_select.models import SelectedFile

def _handle_toggle_all(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    fb_routers: FileBrowserRouters,  # File browser routers
    sess,  # FastHTML session
):  # OOB tuple (selection panel, checkbox OOBs, stats panel)
    """Toggle all media files in the current directory."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])
    extraction_results = step_state.get("extraction_results", {})

    # Get current directory from file browser state
    browser_state = fb_routers._fb_state_getter()
    current_dir = Path(browser_state.current_path)

    # List media files in current directory (shallow)
    media_files = []
    if current_dir.is_dir():
        for child in sorted(current_dir.iterdir()):
            if child.is_file() and is_media_file(str(child)):
                media_files.append(child)

    if not media_files:
        return ()

    # Check if all are already selected
    selected_paths = {f["path"] for f in selected_files}
    media_paths = {str(f) for f in media_files}
    all_selected = media_paths.issubset(selected_paths)

    if all_selected:
        # Remove all media files in current directory
        selected_files = [f for f in selected_files if f["path"] not in media_paths]
        for p in media_paths:
            extraction_results.pop(p, None)
    else:
        # Add unselected media files
        for child in media_files:
            path = str(child)
            if path not in selected_paths:
                file_type = detect_file_type(path)
                if file_type:
                    selected_files.append(SelectedFile(
                        path=path,
                        filename=child.name,
                        file_type=file_type,
                        size_bytes=child.stat().st_size,
                        duration=None,
                        format=child.suffix.lstrip("."),
                    ))

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=selected_files,
        extraction_results=extraction_results,
        verified=False,
        committed_audio_paths=[],
    )

    selected_folders = step_state.get("selected_folders", [])

    selection_oob = render_selection_panel(selected_files, urls, extraction_results)
    selection_oob.attrs["hx-swap-oob"] = "outerHTML"
    checkbox_oobs = _render_oob_checkboxes(fb_routers, selected_files, selected_folders, list(media_paths))
    stats_oob = render_stats_panel(selected_files, urls, extraction_results, verified=False)
    stats_oob.attrs["hx-swap-oob"] = "outerHTML"
    return (selection_oob, *checkbox_oobs, stats_oob)


def init_selection_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider (for OOB browser updates)
    config: FileBrowserConfig,  # Browser configuration (for OOB browser updates)
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    home_path: str = "",  # Home directory path
    fb_routers: FileBrowserRouters = None,  # File browser routers (for OOB sync)
    prefix: str = "/selection",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize selection queue management routes."""
    router = APIRouter(prefix=prefix)

    @router
    def remove(sess, key: str = ""):
        """Remove a file from the selection by key."""
        if not key:
            session_id = get_session_id_from_sess(sess)
            step_state = get_step_state(state_store, workflow_id, session_id)
            return render_selection_panel(step_state.get("selected_files", []), urls)
        return _handle_remove(state_store, workflow_id, urls, fb_routers, sess, key)

    @router
    async def reorder(request, sess):
        """Reorder selected files via SortableJS."""
        return await _handle_reorder(state_store, workflow_id, urls, request, sess)

    @router
    def clear(sess):
        """Clear all selected files."""
        return _handle_clear(state_store, workflow_id, urls, fb_routers, sess)

    @router
    def toggle_all(sess):
        """Toggle all media files in the current directory."""
        return _handle_toggle_all(state_store, workflow_id, urls, fb_routers, sess)

    routes = {
        "remove": remove,
        "reorder": reorder,
        "clear": clear,
        "toggle_all": toggle_all,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()